# ChimeraDB: Getting Started

**Knowledge Graph + Vector Search + SQL Analytics in One DuckDB File**

This notebook shows you how to get started with ChimeraDB in 5 minutes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codimusmaximus/chimeradb/blob/main/examples/getting_started_colab.ipynb)

---

## Installation

In [ ]:
!pip install -q chimeradb duckdb
print("✓ ChimeraDB installed!")

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from chimeradb import KnowledgeGraph

print("✓ Ready to go!")

## Quick Start Example

Let's build a simple knowledge graph with people and companies:

In [ ]:
# Create database with automatic embeddings
kg = KnowledgeGraph(":memory:")
print("✓ Database created with auto-embeddings enabled")

In [ ]:
# Add entities - embeddings automatically generated from 'bio' field
kg.add_entity("alice", {"name": "Alice", "bio": "ML engineer building LLM agents"}, ["Person"], embed_field="bio")
kg.add_entity("bob", {"name": "Bob", "bio": "AI researcher focused on NLP"}, ["Person"], embed_field="bio")
kg.add_entity("acme", {"name": "Acme AI"}, ["Company"])
print("✓ Added 2 people and 1 company")

In [ ]:
# Add relationships
kg.add_relationship("alice", "acme", "WORKS_AT")
kg.add_relationship("bob", "acme", "WORKS_AT")
print("✓ Added employment relationships")

## 1. Semantic Search - Find by MEANING

In [ ]:
results = kg.search("who works on language models?", top_k=2)

print("Results:\n")
for r in results:
    name = r['properties']['name']
    bio = r['properties'].get('bio', 'N/A')
    similarity = r['similarity']
    print(f"  {name}: {bio}")
    print(f"  Similarity: {similarity:.2f}\n")

print("Notice: Found both even though 'language models' doesn't appear in their bios!")
print("That's the power of semantic search with embeddings.")

## 2. Graph Traversal - Follow Relationships

In [ ]:
employees = kg.traverse("acme", direction="incoming", relation_type="WORKS_AT")

print(f"Acme has {len(employees)} employees:\n")
for emp in employees:
    print(f"  - {emp['properties']['name']}")

## 3. SQL/PGQ - Graph Pattern Matching

Use SQL:2023 standard for graph queries:

In [ ]:
results = kg.query("""
    SELECT *
    FROM GRAPH_TABLE (knowledge_graph
        MATCH (p:nodes)-[e:edges]->(c:nodes)
        WHERE c.id = 'acme'
        COLUMNS (
            json_extract_string(p.properties, 'name') as person,
            e.edge_type
        )
    )
""")

print("Relationships:\n")
for person, edge_type in results:
    print(f"  {person} --[{edge_type}]--> Acme AI")

## 4. SQL Analytics - Aggregate Data

In [ ]:
stats = kg.query("""
    SELECT
        json_extract_string(n.properties, 'name') as company,
        COUNT(*) as employee_count
    FROM nodes n
    JOIN edges e ON e.to_id = n.id
    WHERE n.labels LIKE '%Company%'
    GROUP BY company
""")

print("Company Stats:\n")
for company, count in stats:
    print(f"  {company}: {count} employees")

## Summary

You just learned how to:

✅ **Create a knowledge graph** with embeddings  
✅ **Search by meaning** (not just keywords)  
✅ **Traverse relationships** to find connections  
✅ **Use SQL/PGQ** for pattern matching  
✅ **Aggregate data** with SQL analytics  

**All in one DuckDB file!**

---

## Next Steps

Try the **Industrial IoT Example** to see how to:
- Store timeseries data WITHOUT embeddings
- Join knowledge graphs with timeseries tables
- Build production-ready LLM applications

[![Open Industrial IoT Example](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codimusmaximus/chimeradb/blob/main/examples/industrial_iot_colab.ipynb)

## Learn More

- **GitHub**: [github.com/codimusmaximus/chimeradb](https://github.com/codimusmaximus/chimeradb)
- **Documentation**: [ChimeraDB Docs](https://github.com/codimusmaximus/chimeradb/tree/main/docs)
- **More Examples**: [ChimeraDB Examples](https://github.com/codimusmaximus/chimeradb/tree/main/examples)

In [ ]:
kg.close()
print("✅ Done!")